In [1]:
import os
import pandas as pd
import datetime as dt
import pytz
from tqdm import tqdm

In [2]:
print(f'Latest run date: {dt.datetime.today()}')

Latest run date: 2025-01-09 00:32:10.173652


### Constants

In [3]:
# project
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

# task
str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_dirname_output = './output'

Project: 20241112-simple-model-test
Task: 06_join_tsp_data


### Make output directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

### Import data

In [5]:
%%time

str_filename = 'df.gzip'
str_uri = f's3://{str_project}/05_engineer_pmt_hx/{str_filename}'
df = pd.read_parquet(str_uri)
# lower col names
df.columns = [col.lower() for col in df.columns]
# show
df

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:279: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


CPU times: user 8.66 s, sys: 3.2 s, total: 11.9 s
Wall time: 10.7 s


,accountid,request_datetime,response_model_name,file_key,bitdebtor,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,dealerstate__app,...,list_pmt_hx_closed__tu_pmthx,flt_wtd_avg_closed__tu_pmthx,flt_avg_closed__tu_pmthx,int_30dpd_closed__tu_pmthx,int_60dpd_closed__tu_pmthx,int_90dpd_closed__tu_pmthx,int_bad_3mo_closed__tu_pmthx,int_bad_6mo_closed__tu_pmthx,int_bad_3mo_closed_end__tu_pmthx,int_bad_6mo_closed_end__tu_pmthx
0,5702434,2021-07-26 16:29:29.3903686,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,5702434__7162486__20210707,5702434,7162486.0,1,Iowa,...,"[1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0.616667,0.750000,0.0,0.0,0.0,0.0,1.0,1.0,1.0
1,5714239,2021-07-26 16:39:34.1121025,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,5714239__7176826__20210720,5714239,7176826.0,1,Utah,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5713063,2021-07-26 16:48:39.3211104,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,5713063__7175396__20210719,5713063,7175396.0,1,Illinois,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5713732,2021-07-27 09:02:35.3300974,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,5713732__7176216__20210720,5713732,7176216.0,1,Michigan,...,"[1, 1, 1]",1.000000,1.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5715634,2021-07-27 09:18:12.2190097,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,5715634__7178525__20210722,5715634,7178525.0,1,Arizona,...,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1.000000,1.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94474,8420588,2024-11-26 06:16:16+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,8420588103970121,8420588,10397012.0,1,North Carolina,...,"[1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...",0.030636,0.103896,11.0,0.0,0.0,1.0,1.0,1.0,1.0
94475,8401043,2024-11-26 06:21:44+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,8401043103735571,8401043,10373557.0,1,Nevada,...,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, ...",0.367073,0.550000,0.0,0.0,0.0,0.0,0.0,1.0,1.0
94476,8414683,2024-11-26 06:25:09+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,8414683103898941,8414683,10389894.0,1,Virginia,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
94477,8359085,2024-11-26 06:32:28+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,8359085103232431,8359085,10323243.0,1,Alabama,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Drop some columns because they are already in TSP data

In [6]:
# list cols to drop because they are already in tsp
list_cols = [
    'uniqueid__app',
    'bigaccountid__app',
    'dtmfunded__app',
    'bigdebtorid__app',
    'dtmstampcreation__app',
    'intterm__app',
    'fltdowncash__app',
    'fltapproveddowntotal__app',
    'bitservicecontract__app',
    'applicationdate__app',
    'monthonbooks__app',
    'runningnetloss__app',
    'amtfinanced__app',
    'intopenbktype__app',
    'vehicleyear__app',
    'bitnew__app',
    'vehiclemake__app',
    'miles_odometer__app',
    'bookvalue__app',
    'payment__app',
    'dti__app',
    'pti__app',
    'fltadvance__app',
    'strvehicletype__app',
    'bitgap__app',
    'dealerstampcreation__app',
]
list_cols = [col for col in list_cols if col in list(df.columns)]
# drop
df.drop(list_cols, axis=1, inplace=True)
# show
df

,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,list_pmt_hx_closed__tu_pmthx,flt_wtd_avg_closed__tu_pmthx,flt_avg_closed__tu_pmthx,int_30dpd_closed__tu_pmthx,int_60dpd_closed__tu_pmthx,int_90dpd_closed__tu_pmthx,int_bad_3mo_closed__tu_pmthx,int_bad_6mo_closed__tu_pmthx,int_bad_3mo_closed_end__tu_pmthx,int_bad_6mo_closed_end__tu_pmthx
0,5702434,2021-07-26 16:29:29.3903686,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,"[1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0.616667,0.750000,0.0,0.0,0.0,0.0,1.0,1.0,1.0
1,5714239,2021-07-26 16:39:34.1121025,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5713063,2021-07-26 16:48:39.3211104,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5713732,2021-07-27 09:02:35.3300974,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Michigan,Independent,Michigan,False,...,"[1, 1, 1]",1.000000,1.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5715634,2021-07-27 09:18:12.2190097,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Arizona,Franchise,Arizona,False,...,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1.000000,1.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94474,8420588,2024-11-26 06:16:16+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,North Carolina,Independent,North Carolina,True,...,"[1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...",0.030636,0.103896,11.0,0.0,0.0,1.0,1.0,1.0,1.0
94475,8401043,2024-11-26 06:21:44+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Nevada,Franchise,Nevada,True,...,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, ...",0.367073,0.550000,0.0,0.0,0.0,0.0,0.0,1.0,1.0
94476,8414683,2024-11-26 06:25:09+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Virginia,Franchise,Virginia,False,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
94477,8359085,2024-11-26 06:32:28+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Alabama,Franchise,Alabama,True,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Get the data from TSP

In [7]:
%%time

str_filename = 'df_tsp.gzip'
str_uri = f's3://{str_project}/02_get_tsp_data/{str_filename}'
df_tmp = pd.read_parquet(str_uri)
# rename
dict_rename = {
    'bigaccountid__app': 'accountid',
    'dtmstampcreation__app': 'applicationdate__app',
}
df_tmp.rename(columns=dict_rename, inplace=True)
# convert to dtm
list_cols = [
    'applicationdate__app',
    'dtmfunded__app',
    'dealerstampcreation__app',
]
for col in tqdm(list_cols):
    df_tmp[col] = pd.to_datetime(df_tmp[col])
# show
df_tmp

100%|██████████| 3/3 [00:00<00:00, 38.29it/s]

CPU times: user 266 ms, sys: 40.8 ms, total: 307 ms
Wall time: 472 ms


,uniqueid__app,accountid,bigdebtorid__app,bitdebtor__app,applicationdate__app,dtmfunded__app,bitdefault__app,monthonbooks__app,runningnetloss__app,amtfinanced__app,...,fltapproveddowntotal__app,payment__app,dti__app,pti__app,bitservicecontract__app,fltadvance__app,strvehicletype__app,bitgap__app,dealerstampcreation__app,totaldebt__app
0,546593468846511,5465934,6884651,1,2021-01-01 09:41:26.617,2021-01-07,0,48,0.0,15779.71,...,0.0,362.51,0.249783,0.057318,0,1.192262,auto,1,2020-10-23 13:11:54.257,1091.00
1,546599768847241,5465997,6884724,1,2021-01-01 11:13:07.147,2021-01-11,0,48,0.0,16545.58,...,2160.0,381.96,0.327537,0.119892,1,0.898571,auto,1,2020-02-07 08:29:35.927,661.53
2,546604668847831,5466046,6884783,1,2021-01-01 12:26:26.883,2021-01-18,0,48,0.0,24572.00,...,0.0,556.62,0.383475,0.113944,1,1.118690,suv,0,2014-03-14 16:04:13.850,1316.67
3,546611768848671,5466117,6884867,1,2021-01-01 13:40:20.980,2021-01-20,0,48,0.0,16239.90,...,4000.0,384.75,0.377446,0.091937,1,1.104110,van,1,2020-06-17 10:05:47.617,1194.83
4,546611868848681,5466118,6884868,1,2021-01-01 13:41:09.923,2021-01-14,0,48,0.0,22420.92,...,0.0,508.41,0.464816,0.081186,1,1.125723,suv,1,2017-04-07 09:23:49.460,1850.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101948,8535032105278331,8535032,10527834,0,2024-12-31 16:04:16.250,2025-01-02,0,None,NaN,34991.50,...,0.0,1022.20,0.368418,0.092049,0,1.133690,suv,1,2022-03-04 11:38:51.370,19.86
101949,8536877105299991,8536877,10530000,0,2025-01-02 11:38:41.897,2025-01-07,0,None,NaN,30840.40,...,0.0,825.27,0.450806,0.122711,0,1.035246,auto,1,2022-08-19 14:38:01.340,836.48
101950,8538151105315191,8538151,10531520,0,2025-01-02 16:03:32.067,2025-01-06,0,None,NaN,16089.34,...,500.0,463.58,0.279084,0.079084,1,1.103788,suv,1,2022-03-29 16:06:09.463,NaN
101951,8543705105382391,8543705,10538240,0,2025-01-04 14:39:02.697,2025-01-07,0,None,NaN,18339.08,...,5200.0,538.27,0.499661,0.115027,0,0.993146,auto,1,2022-11-28 15:22:35.180,864.00


### Set dtypes for joining

In [8]:
# payloads
df['accountid'] = df['accountid'].astype(int)
df['bitdebtor__app'] = df['bitdebtor__app'].astype(int)
# tsp
df_tmp['accountid'] = df_tmp['accountid'].astype(int)
df_tmp['bitdebtor__app'] = df_tmp['bitdebtor__app'].astype(int)

### Join

In [9]:
%%time

# join
df = pd.merge(
    left=df,
    right=df_tmp,
    left_on=['accountid','bitdebtor__app'],
    right_on=['accountid','bitdebtor__app'],
    how='inner',
)
# save memory
del df_tmp
# show
df

CPU times: user 776 ms, sys: 769 ms, total: 1.54 s
Wall time: 1.54 s


,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,fltapproveddowntotal__app,payment__app,dti__app,pti__app,bitservicecontract__app,fltadvance__app,strvehicletype__app,bitgap__app,dealerstampcreation__app,totaldebt__app
0,5702434,2021-07-26 16:29:29.3903686,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,2000.0,344.95,0.494484,0.142150,0,1.150000,auto,1,2012-05-30 10:50:45.437,855.00
1,5714239,2021-07-26 16:39:34.1121025,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,1000.0,598.14,0.363221,0.102956,1,1.000289,auto,1,2018-12-21 12:37:32.413,1512.06
2,5713063,2021-07-26 16:48:39.3211104,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,0.0,418.11,0.414040,0.111234,1,1.149861,auto,1,2010-01-25 15:07:34.910,1138.20
3,5713732,2021-07-27 09:02:35.3300974,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Michigan,Independent,Michigan,False,...,0.0,679.78,0.380767,0.125498,0,0.993292,suv,1,2015-05-04 16:04:37.317,1382.71
4,5715634,2021-07-27 09:18:12.2190097,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Arizona,Franchise,Arizona,False,...,0.0,399.64,0.418160,0.083485,1,0.945898,auto,1,2012-12-27 10:14:39.477,1602.08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94472,8420588,2024-11-26 06:16:16+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,North Carolina,Independent,North Carolina,True,...,1000.0,848.97,0.319642,0.104996,1,1.096184,suv,1,2016-08-30 13:05:21.300,1735.57
94473,8401043,2024-11-26 06:21:44+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Nevada,Franchise,Nevada,True,...,500.0,699.46,0.319354,0.119353,1,0.955559,suv,1,2024-09-06 11:26:41.533,1172.09
94474,8414683,2024-11-26 06:25:09+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Virginia,Franchise,Virginia,False,...,0.0,445.28,0.286484,0.086483,0,1.152536,auto,1,2018-09-06 15:55:22.703,1029.75
94475,8359085,2024-11-26 06:32:28+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Alabama,Franchise,Alabama,True,...,2200.0,586.59,0.451532,0.115865,0,0.934501,auto,0,2013-06-28 09:08:19.850,1699.38


### Remove duplicates

In [10]:
df.drop_duplicates(subset=['accountid','bitdebtor__app'], keep='last', inplace=True)
# show
df

,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,fltapproveddowntotal__app,payment__app,dti__app,pti__app,bitservicecontract__app,fltadvance__app,strvehicletype__app,bitgap__app,dealerstampcreation__app,totaldebt__app
0,5702434,2021-07-26 16:29:29.3903686,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,2000.0,344.95,0.494484,0.142150,0,1.150000,auto,1,2012-05-30 10:50:45.437,855.00
1,5714239,2021-07-26 16:39:34.1121025,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,1000.0,598.14,0.363221,0.102956,1,1.000289,auto,1,2018-12-21 12:37:32.413,1512.06
2,5713063,2021-07-26 16:48:39.3211104,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,0.0,418.11,0.414040,0.111234,1,1.149861,auto,1,2010-01-25 15:07:34.910,1138.20
3,5713732,2021-07-27 09:02:35.3300974,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Michigan,Independent,Michigan,False,...,0.0,679.78,0.380767,0.125498,0,0.993292,suv,1,2015-05-04 16:04:37.317,1382.71
4,5715634,2021-07-27 09:18:12.2190097,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Arizona,Franchise,Arizona,False,...,0.0,399.64,0.418160,0.083485,1,0.945898,auto,1,2012-12-27 10:14:39.477,1602.08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94472,8420588,2024-11-26 06:16:16+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,North Carolina,Independent,North Carolina,True,...,1000.0,848.97,0.319642,0.104996,1,1.096184,suv,1,2016-08-30 13:05:21.300,1735.57
94473,8401043,2024-11-26 06:21:44+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Nevada,Franchise,Nevada,True,...,500.0,699.46,0.319354,0.119353,1,0.955559,suv,1,2024-09-06 11:26:41.533,1172.09
94474,8414683,2024-11-26 06:25:09+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Virginia,Franchise,Virginia,False,...,0.0,445.28,0.286484,0.086483,0,1.152536,auto,1,2018-09-06 15:55:22.703,1029.75
94475,8359085,2024-11-26 06:32:28+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Alabama,Franchise,Alabama,True,...,2200.0,586.59,0.451532,0.115865,0,0.934501,auto,0,2013-06-28 09:08:19.850,1699.38


### Write to s3

In [11]:
%%time

str_filename = 'df.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df.to_parquet(str_uri, compression='gzip')

CPU times: user 42.3 s, sys: 218 ms, total: 42.5 s
Wall time: 46.3 s
